# Bifrost Getting Started — Groq + Mistral + Jina + Qdrant Cloud

### A free-tier, self-hosted AI gateway walkthrough

---

**Purpose:** Get hands-on with [Bifrost](https://github.com/maximhq/bifrost) (self-hosted, open-source AI
gateway by Maxim AI) using only providers with usable free tiers:

- **Groq** — chat/completions (LPU inference, very fast, generous free tier)
- **Mistral** — chat/completions (La Plateforme free tier)
- **Jina AI** — embeddings (used only in the RAG section, called directly — not via Bifrost)
- **Qdrant Cloud** — vector store (managed, used only in the RAG section)

**Your setup:** a single, already-created Docker container named `bifrost`, started with
`docker start bifrost` and nothing else. Everything Bifrost-side is configured through its
**Web UI** at `http://localhost:8080`, which persists inside that container across restarts.

**What you'll cover:**
1. Baseline: vanilla Groq/Mistral SDKs (no gateway)
2. Bifrost setup via the Web UI & a drop-in client
3. Bifrost feature showcase: unified API, failover, load balancing, streaming, observability, virtual keys, MCP
4. A small RAG pipeline built by hand (Jina embeddings + Qdrant Cloud + Bifrost generation)

**Prerequisites:**
1. Python 3.10+ and packages from `requirements.txt` (`pip install -r requirements.txt`)
2. `.env` filled in — `GROQ_API_KEY`, `MISTRAL_API_KEY` required from Section 1 onward; `JINA_API_KEY`,
   `QDRANT_URL`, and `QDRANT_API_KEY` required only for Section 4
3. `docker start bifrost` — that's it, nothing else to run
4. Groq + Mistral added as providers in the Bifrost Web UI (Section 2.2 walks through this)

**A note on style:** every code cell here does one small thing. Where the same handful of lines
would otherwise repeat across features (timing a call, reading which provider answered), there's
one small helper function instead — defined once, reused everywhere. Full term definitions live
in [`docs/00-terminologies.md`](docs/00-terminologies.md).

---


---
<a id="section-0"></a>
## Section 0: Environment Setup & Prerequisites


LOAD YOUR ENV VARS

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [17]:
os.getenv("BIFROST_GROQ_VIRTUAL_KEY")

'sk-bf-fd12af98-e5cc-45f8-879b-b09159d10d3e'

In [12]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
BIFROST_BASE_URL = os.getenv("BIFROST_BASE_URL", "http://localhost:8080")
BIFROST_GROQ_VIRTUAL_KEY = os.getenv("BIFROST_GROQ_VIRTUAL_KEY", "")
BIFROST_MISTRAL_VIRTUAL_KEY = os.getenv("BIFROST_MISTRAL_VIRTUAL_KEY", "")

print("Keys loaded. Bifrost base URL:", BIFROST_BASE_URL)

Keys loaded. Bifrost base URL: http://localhost:8080


### 0.3 Bifrost Health Check

A quick check that the `bifrost` container is actually up before anything else tries to call it.


In [3]:
import httpx 

try : 
    health = httpx.get(f"{BIFROST_BASE_URL}/health" , timeout = 3)
    bifrost_ok = health.status_code == 200
except Exception:
    bifrost_ok = False
    
print("Bifrost:", "Online" if bifrost_ok else "Offline")

if not bifrost_ok:
    print("Start it with: docker start bifrost")    
    

Bifrost: Online


### 0.4 Prompts We'll Reuse

One set of prompts, used throughout the notebook, so the vanilla SDKs, Bifrost, and the RAG
section are all answering the same questions.


In [4]:
TEST_PROMPTS = {
    "simple": "What is the capital of France?",
    "reasoning": "Explain the difference between RAG and fine-tuning in 3 bullet points.",
    "code": "Write a Python function that validates an email address using regex.",
    "duplicate1": "What is LangChain used for?",
    "duplicate2": "What is LangChain primarily used for?",
    "deepwiki": "What are the stream modes in the new langgraph version? Use the deepwiki tool to check the langchain-ai/langgraph repo.",
    "tavily": "Search the web for the latest news about Groq AI and summarize the top result.",
}

### 0.5 Models We'll Use

Groq is retiring `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` on 2026-08-16 (free tier),
so this notebook uses their recommended replacements instead. Mistral's `-latest` aliases always
point at the current model, so they need no such swap.

`openai/gpt-oss-120b` is OpenAI's *open-weight* model, served on Groq's own hardware — it runs on
`GROQ_API_KEY`, not an OpenAI key. Nothing here touches your OpenAI account.


In [13]:
import time 


# GROQ MODELS 
GROQ_MODEL = "openai/gpt-oss-120b"
GROQ_MODEL_FALLBACK = "openai/gpt-oss-120b"

MISTRAL_MODEL="codestral-latest"
MISTRAL_MODEL_FALLBACK="mistral-small-latest"

print("Groq:", GROQ_MODEL, "| fallback:", GROQ_MODEL_FALLBACK)
print("Mistral:", MISTRAL_MODEL, "| fallback:", MISTRAL_MODEL_FALLBACK)

Groq: openai/gpt-oss-120b | fallback: openai/gpt-oss-120b
Mistral: codestral-latest | fallback: mistral-small-latest


---
<a id="section-1"></a>
## Section 1: Baseline — Vanilla Groq & Mistral SDKs

> **Goal:** show what you have to write yourself WITHOUT a gateway — two separate SDKs, two
> separate error-handling paths, no shared observability.


In [6]:
from groq import Groq 

client_groq= Groq(api_key=GROQ_API_KEY)

In [7]:
start = time.perf_counter()
response = client_groq.chat.completions.create(
    model = GROQ_MODEL,
    messages=[
        {
            "role":"user",
            "content" : TEST_PROMPTS["simple"]
        }
    ]
)
latency_ms = (time.perf_counter() - start) * 1000

print(response.choices[0].message.content)
print(f"\n{latency_ms:.0f} ms, {response.usage.total_tokens} tokens")



The capital of France is **Paris**.

448 ms, 126 tokens


In [8]:
from mistralai.client import Mistral

client_mistral = Mistral(api_key=MISTRAL_API_KEY)


start = time.perf_counter()
response = client_mistral.chat.complete(
    model=MISTRAL_MODEL,
    messages=[{"role": "user", "content": TEST_PROMPTS["reasoning"]}],
)
latency_ms = (time.perf_counter() - start) * 1000

print(response.choices[0].message.content)
print(f"\n{latency_ms:.0f} ms, {response.usage.total_tokens} tokens")


Here’s a concise comparison between **RAG (Retrieval-Augmented Generation)** and **fine-tuning** in three key points:

1. **Approach**:
   - **RAG** combines a retrieval step (fetching relevant documents) with a generative model (e.g., LLM) to produce answers, without altering the model’s weights.
   - **Fine-tuning** updates the model’s weights on task-specific data to improve performance, often requiring retraining.

2. **Flexibility vs. Specialization**:
   - **RAG** is adaptable to new knowledge by updating the retrieval corpus, keeping the model unchanged.
   - **Fine-tuning** specializes the model for a specific task but may struggle with new or unseen data without further updates.

3. **Resource Usage**:
   - **RAG** relies on external knowledge sources (e.g., databases) and may require more compute for retrieval.
   - **Fine-tuning** demands significant compute and data to train, but once done, inference is efficient.

In short, RAG leverages external knowledge dynamically, whi

### 1.3 Vanilla Streaming — Groq

For chat UIs that render tokens as they arrive, instead of waiting for the full response. Shown
here on the raw SDK so Section 3.4 can show the same thing through Bifrost.


In [9]:
stream = client_groq.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": TEST_PROMPTS["code"]}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end ="" , flush=True)
print()

Below is a self‑contained Python snippet that defines a single function **`is_valid_email`**.  
It uses the standard library’s `re` module and a regular expression that covers the most common (and RFC‑5322‑compatible) rules for an email address while staying readable and maintainable.

```python
import re
from typing import Pattern

# ----------------------------------------------------------------------
# Pre‑compiled regex (compiled once, reused on every call)
# ----------------------------------------------------------------------
_EMAIL_REGEX: Pattern = re.compile(
    r"""^
    # Local part
    (?:[a-zA-Z0-9!#$%&'*+/=?^_`{|}~-]+          # 1+ allowed chars
        (?:\.[a-zA-Z0-9!#$%&'*+/=?^_`{|}~-]+)* # optional dot‑separated sections
    |
        "(?:[\x01-\x08\x0b\x0c\x0e-\x1f\x21\x23-\x5b\x5d-\x7f]|
           \\[\x01-\x09\x0b\x0c\x0e-\x7f])*"   # quoted string
    )
    @
    # Domain part
    (?:                                          # either:
        (?:[a-zA-Z0-9]     

### 1.4 What's Missing Without a Gateway

| Problem                   | Vanilla Groq SDK | Vanilla Mistral SDK |
|---------------------------|-------------------|-----------------------|
| Automatic failover        | Manual            | Manual                |
| Multi-provider routing    | Not possible      | Not possible           |
| Unified request/response  | Per-SDK           | Per-SDK                |
| Rate limit handling       | Manual retry      | Manual retry           |
| Observability / tracing   | External only     | External only          |
| Budget enforcement        | Not possible      | Not possible            |
| MCP tool governance        | Not possible      | Not possible            |
| Streaming normalization   | Per-provider       | Per-provider            |


### 1.5 Manual Fallback — What You'd Have to Write Yourself

The fallback logic teams end up hand-writing without a gateway: try Groq, and if it fails, retry
on Mistral. Section 3.2 replaces this whole function with a single HTTP header.


In [10]:
def naive_manual_fallback(prompt:str) -> str:
    try:
        response = client_groq.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content
    except Exception as groq_error:
        print("GROQ FAILED , trying mistral as backup:" , groq_error)
        response = client_mistral.chat.complete(
            model=MISTRAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content


In [11]:
answer = naive_manual_fallback(TEST_PROMPTS["simple"])
print(answer)

The capital of France is **Paris**.


---
<a id="section-2"></a>
## Section 2: Bifrost Setup & Architecture

> **Bifrost** is a Go-based, open-source AI gateway. It provides failover, load balancing,
> virtual keys, an MCP gateway, and observability — all via a single self-hosted container.


### 2.1 Your Bifrost Container

You already have a container named `bifrost`. The full lifecycle for this notebook is just:

```bash
docker start bifrost   # bring it up
docker stop bifrost    # shut it down when you're done
```

No `docker run`, no `docker-compose`, no extra containers. Bifrost persists whatever you
configure through its Web UI **inside that same container**, so it survives stop/start cycles.

**Do you need a Bifrost API key?** No. Bifrost doesn't require its own credential for incoming
requests by default — that's Virtual Keys (Section 3.6), an optional layer you can turn on later.
The client below is the *unscoped* drop-in client: it needs to reach both Groq and Mistral, so it
uses a placeholder instead of a virtual key (a virtual key is scoped to whatever provider it was
created for — Section 3.6 uses `BIFROST_GROQ_VIRTUAL_KEY` and `BIFROST_MISTRAL_VIRTUAL_KEY`
directly, one client per provider).


### 2.2 Add Groq & Mistral as Providers (Web UI)

1. Open `http://localhost:8080`
2. Go to **Providers** → **Add Provider**
3. Provider = `groq`, paste your actual `GROQ_API_KEY` value directly into the key field
4. Repeat for provider = `mistral` with your `MISTRAL_API_KEY`
5. (Optional, for Section 3.7) Go to **MCP Gateway → MCP Catalog** → **Add MCP Server**:
   - `deepwiki` — connection type `HTTP (Streamable)`, URL `https://mcp.deepwiki.com/mcp`, auth `None`
   - `tavily` — same connection type, URL `https://mcp.tavily.com/mcp/?tavilyApiKey=YOUR_TAVILY_KEY`
     (free key from `https://app.tavily.com/home`), auth `None`
   - **Important:** open each one and toggle **Auto-execute** on for its tools. Without this,
     Bifrost proposes the tool call and stops — it won't actually run it.


### 2.3 Bifrost Drop-in Client Setup

One `OpenAI` client instance now talks to every provider Bifrost knows about — which provider
handles a call is picked by the `"<provider>/<model>"` string, not by which client you made.


In [12]:
from openai import OpenAI 

bifrost = OpenAI(
    api_key = "dont need because we have virtual keys",
    base_url = f"{BIFROST_BASE_URL}/v1"
)

print("Bifrost client ready. Pick a provider via the model string, e.g.:")
print(f"  groq/{GROQ_MODEL}")
print(f"  mistral/{MISTRAL_MODEL}")


Bifrost client ready. Pick a provider via the model string, e.g.:
  groq/openai/gpt-oss-120b
  mistral/codestral-latest


---
<a id="section-3"></a>
## Section 3: Bifrost Feature Showcase


### 3.0 One Helper We'll Reuse

Every feature below calls Bifrost the same way: send a prompt, get back the answer plus *which
provider* actually handled it. Bifrost injects that as a top-level `extra_fields` key in the raw
JSON response — outside the normal OpenAI response schema, so we go through
`with_raw_response` and parse the JSON ourselves to read it, rather than trust an SDK attribute
to carry through a field it was never designed to know about.


In [28]:
import json as _json


def call_bifrost(prompt: str, model: str, fallback_model: str = None):
    headers = {"x-bifrost-fallback-models": fallback_model} if fallback_model else None

    try:
        raw = bifrost.chat.completions.with_raw_response.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            extra_headers=headers,
        )
        response = raw.parse()
        provider = _json.loads(raw.text).get("extra_fields", {}).get("provider", "unknown")
        return response.choices[0].message.content, provider
    except Exception as e:
        msg = str(e).lower()
        if (
            "dashboard auth is not configured" in msg
            or "setup token" in msg
            or "unauthorized" in msg
            or "401" in msg
            or "not configured" in msg
        ):
            raise RuntimeError(
                "Bifrost setup is not complete. Open http://localhost:8080, finish the setup token flow, then add the groq and mistral providers in the dashboard."
            ) from e
        if "failed to get config for provider" in msg or (
            "provider" in msg and "not found" in msg
        ) or "not found" in msg:
            raise RuntimeError(
                "Bifrost is running but the provider is missing. Open http://localhost:8080 and add groq/mistral providers with valid API keys."
            ) from e
        raise


### 3.1 Feature 1 — Unified Drop-in API (Groq + Mistral, Same Client)

Swap providers per-request — cheap/fast Groq for simple queries, Mistral for others — without
maintaining two SDKs or two code paths.


In [31]:
answer , provider = call_bifrost(TEST_PROMPTS["simple"] , f"groq/{GROQ_MODEL}")
print(f"[{provider}] {answer}")

[groq] The capital of France is **Paris**.


In [32]:
answer , provider = call_bifrost( "how to learn ai gateways shortcut , respond in 10 words" , f"mistral/{MISTRAL_MODEL}")
print(f"[{provider}] {answer}")

[mistral] Practice daily, focus on patterns, use mnemonics, review regularly, and apply in real tests.


### 3.2 Feature 2 — Automatic Failover / Provider Fallback

Keep serving requests even if your primary provider is down or rate-limiting you. Bifrost retries
on the fallback model automatically — your code never sees the failure. It triggers on **5xx,
429, and 401**; it does **not** trigger on a **404** (model not found) — that's a bug in your
request, not an outage, so Bifrost passes the error straight through instead of masking it.


In [41]:
answer , provider = call_bifrost(
    TEST_PROMPTS["duplicate2"] , f"groq/{GROQ_MODEL}",
    fallback_model=f"mistral/{MISTRAL_MODEL}"
)
print(f"[{provider}] {answer}")

[groq] **LangChain** is a development framework designed to make it easy to build **applications that are powered by large language models (LLMs)**. Its primary purpose is to provide the building blocks and abstractions needed to:

| Core Capability | What It Lets You Do |
|-----------------|----------------------|
| **Prompt Management** | Define, reuse, and dynamically modify prompts (templates, few‑shot examples, chat histories). |
| **Chains** | Connect multiple LLM calls, utilities, or APIs into a sequential workflow (e.g., “extract‑then‑summarize”, “search‑then‑answer”). |
| **Agents** | Give the LLM the ability to decide which tools (search, calculators, code execution, etc.) to invoke during a conversation. |
| **Memory** | Persist context across turns so the model can “remember” prior user inputs, system states, or external data. |
| **Data Integration** | Load, index, and retrieve information from documents, databases, APIs, or vector stores to augment LLM reasoning. |
| **Ev

In [ ]:
# Bad model name (404) — does NOT trigger the Mistral fallback, by design
try:
    answer, provider = call_bifrost(
    TEST_PROMPTS["duplicate2"] , 
    f"groq/fake-model",
    fallback_model=f"mistral/{MISTRAL_MODEL}"
    )
    print(f"[{provider}] {answer}")
except Exception as e:
    print("No fallback triggered, as expected for a 404:", e)


No fallback triggered, as expected for a 404: Error code: 404 - {'is_bifrost_error': False, 'status_code': 404, 'error': {'type': 'invalid_request_error', 'code': 'model_not_found', 'message': 'The model `fake-model` does not exist or you do not have access to it.'}, 'extra_fields': {'routing_info': {'provider': 'groq', 'model': 'fake-model', 'key': 'new_KEY_now'}, 'provider': 'groq', 'original_model_requested': 'fake-model', 'resolved_model_used': 'fake-model', 'request_type': 'chat_completion', 'latency': 17}}


### 3.3 Feature 3 — Weighted Load Balancing Across API Keys

Bifrost spreads traffic across multiple keys for the *same* provider, by weight, to raise your
effective rate limit. A single request can't show that — it can only ever hit one key. So this
fires several requests and tallies which key actually handled each one. With only one Groq key
configured (the default here), every request will land on that same key — the distribution only
gets interesting once you add a second key with its own weight in the Web UI.


In [34]:
from collections import Counter

# 10 - 5 , 5 

def bifrost_key_used(model: str, prompt: str = "Hello") -> str:
    response = httpx.post(
        f"{BIFROST_BASE_URL}/v1/chat/completions",
        json={"model": model, "messages": [{"role": "user", "content": prompt}]},
        timeout=30,
    )
    return response.json().get("extra_fields", {}).get("routing_info", {}).get("key", "unknown")


In [35]:
counts = Counter(bifrost_key_used(f"groq/{GROQ_MODEL}") for _ in range(10))
print("Requests per key (10 calls):", dict(counts))

Requests per key (10 calls): {'groq': 10}


### 3.4 Feature 4 — Streaming (Unified Across Providers)

Same streaming function, no `if provider == "groq"` branching — swap the model string and the
same code streams from a different backend.


In [36]:
def stream_bifrost(prompt: str, model: str):
    stream = bifrost.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()


In [37]:
stream_bifrost(TEST_PROMPTS["code"], f"groq/{GROQ_MODEL}")

Below is a self‑contained Python function that uses a regular‑expression to check whether a string looks like a valid e‑mail address.

```python
import re

# ------------------------------------------------------------
# Email‑validation helper
# ------------------------------------------------------------
def is_valid_email(email: str) -> bool:
    """
    Return True if *email* matches a reasonable e‑mail address pattern,
    otherwise False.

    The pattern follows the most common rules from RFC 5322 while staying
    readable and fast:

      * Local‑part may contain letters, digits, and the characters
        . _ % + - (dot may not be the first or last character and
        cannot appear consecutively).
      * Domain consists of one or more labels separated by a single dot.
        Each label may contain letters, digits, and hyphens, but may not
        start or end with a hyphen.
      * The top‑level domain must be at least two characters long.

    The function deliberately a

In [38]:
stream_bifrost(TEST_PROMPTS["code"], f"mistral/{MISTRAL_MODEL}")

# Python Function to Validate Email Address Using Regex

Here's a Python function that validates an email address using regular expressions:

```python
import re

def validate_email(email):
    """
    Validates an email address using regex pattern.

    Args:
        email (str): The email address to validate

    Returns:
        bool: True if the email is valid, False otherwise
    """
    # Regular expression pattern for email validation
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    # Check if the email matches the pattern
    if re.fullmatch(pattern, email):
        return True
    return False

# Example usage
if __name__ == "__main__":
    test_emails = [
        "user@example.com",       # Valid
        "firstname.lastname@example.com",  # Valid
        "user+tag@example.org",   # Valid
        "user.name@example.co.uk", # Valid
        "invalid.email@",          # Invalid
        "invalid@.com",            # Invalid
        "@missingusername.com",    #

### 3.5 Feature 5 — Observability: Logs API

See exactly what Bifrost routed, to which provider, and how long it took — without adding any
logging of your own. Full Prometheus `/metrics` needs a separate plugin binary (not covered
here); this REST endpoint works out of the box.


In [42]:
import pprint

response = httpx.get(f"{BIFROST_BASE_URL}/api/logs?limit=5", timeout=5)
logs = response.json() if response.status_code == 200 else {"status": response.status_code}

pprint.pprint(logs)


{'status': 401}


### 3.6 Feature 6 — Virtual Keys & Budget Governance

Hand a teammate, or a specific app, a scoped key with a spending cap — instead of giving out your
raw Groq/Mistral keys. Create one at `http://localhost:8080` → **Virtual Keys** (a separate tab
from **Providers**) for each provider you want to govern separately. Bifrost shows you the
generated secret once — put it in `.env` as `BIFROST_GROQ_VIRTUAL_KEY` /
`BIFROST_MISTRAL_VIRTUAL_KEY` to try these cells. Each key only works for the provider it was
created for, so unlike Section 2.3's client, this is one client per virtual key.


In [43]:
def call_with_virtual_key(virtual_key: str, model: str, prompt: str):
    if not virtual_key:
        print("No virtual key set for this provider — create one at http://localhost:8080 -> Virtual Keys")
        return
    client = OpenAI(api_key=virtual_key, base_url=f"{BIFROST_BASE_URL}/v1")
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print("Call failed — check the virtual key is valid and not over budget:", e)


In [58]:
BIFROST_GROQ_VIRTUAL_KEY ="sk-bf-c4360e7e-c80f-40b9-a0a0-7d21cd0bf1c0"
BIFROST_MISTRAL_VIRTUAL_KEY= "sk-bf-6532a032-6c56-4645-8127-bd1592cfdc5a"

In [59]:
call_with_virtual_key(BIFROST_GROQ_VIRTUAL_KEY, f"groq/{GROQ_MODEL}", TEST_PROMPTS["simple"])

The capital of France is **Paris**.


In [60]:
call_with_virtual_key(BIFROST_MISTRAL_VIRTUAL_KEY, f"mistral/{MISTRAL_MODEL}", TEST_PROMPTS["code"])

# Python Function to Validate Email Address Using Regex

Here's a Python function that validates an email address using regular expressions:

```python
import re

def validate_email(email):
    """
    Validates an email address using regex pattern.

    Args:
        email (str): The email address to validate

    Returns:
        bool: True if the email is valid, False otherwise
    """
    # Regular expression pattern for email validation
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

    # Compile the pattern and match against the email
    if re.fullmatch(pattern, email):
        return True
    return False

# Example usage
if __name__ == "__main__":
    test_emails = [
        "user@example.com",        # Valid
        "firstname.lastname@example.com",  # Valid
        "user+tag@example.com",    # Valid
        "user@subdomain.example.com",  # Valid
        "invalid.email@",           # Invalid
        "user@.com",                 # Invalid
        "@example.

### 3.7 Feature 7 — MCP Gateway (DeepWiki + Tavily)

Let the model call external tools — DeepWiki (ask questions about a GitHub repo) and Tavily (live
web search) — through the gateway, instead of wiring tool-calling into every client app
separately. Requires both MCP clients added with **Auto-execute** on (Section 2.2).

> **Model note:** Groq's `openai/gpt-oss-*` models fail with a 400 during Bifrost's tool-calling
> loop (`'reasoning_content' is unsupported` — Bifrost resends the model's own reasoning trace,
> and Groq's API rejects it for these models). So this section runs on **Mistral** instead, which
> has no such issue.


REGISTOR MCP TO BIFROST

In [ ]:
response = httpx.get(f"{BIFROST_BASE_URL}/api/mcp/clients",
    auth=("admin", "1"),
 timeout=5)
response






<Response [200 OK]>

In [65]:
clients = response.json().get("clients", [])

clients

[{'config': {'allow_on_all_virtual_keys': False,
   'client_id': 'b80a9c3e-f7ef-47ef-9938-0f9af084262a',
   'name': 'deepwiki',
   'endpoint_slug': 'deepwiki',
   'is_code_mode_client': False,
   'connection_type': 'http',
   'connection_string': {'value': 'https://mcp.deepwiki.com/mcp',
    'type': 'plain_text'},
   'tls_config': {},
   'auth_type': 'headers',
   'tools_to_execute': ['*'],
   'tools_to_auto_execute': ['*'],
   'is_ping_available': True,
   'needs_session_stickiness': False,
   'disabled': False,
   'allow_by_default': False},
  'tools': [{'name': 'ask_wiki_question',
    'description': "Ask any question about a GitHub repository's codebase and get an AI-powered answer\ngrounded in its DeepWiki.",
    'parameters': {'type': 'object',
     'properties': {'question': {'description': 'The question to ask about the repository.',
       'type': 'string'},
      'repoName': {'anyOf': [{'type': 'string'},
        {'items': {'type': 'string'}, 'type': 'array'}],
       'descri

In [66]:
registered = {c["config"]["name"] for c in clients}

In [67]:
print("Registered:", registered or "none")

Registered: {'tavily', 'deepwiki'}


In [87]:
def call_with_tools(prompt: str, servers: list[str]) -> dict:
    BIFROST_GROQ_VIRTUAL_KEY ="sk-bf-c4360e7e-c80f-40b9-a0a0-7d21cd0bf1c0"
    BIFROST_MISTRAL_VIRTUAL_KEY= "sk-bf-6532a032-6c56-4645-8127-bd1592cfdc5a"
    response = httpx.post(
    f"{BIFROST_BASE_URL}/v1/chat/completions",
    headers={"Authorization": f"Bearer {BIFROST_MISTRAL_VIRTUAL_KEY}"},
    json={
        "model": f"mistral/{MISTRAL_MODEL}",
        "messages": [{"role": "user", "content": prompt}],
        "bifrost": {
            "mcp": {
                "enabled": True,
                "code_mode": True,
                "servers": servers,
            }
        },
    },
    timeout=60,
)
    # print(response.json())
    return response.json()["choices"][0]["message"]

In [88]:
call_with_tools(TEST_PROMPTS["deepwiki"], ["deepwiki"])

{'role': 'assistant',
 'content': "To determine the stream modes in the new `langgraph` version, I'll use the `deepwiki` tool to check the `langchain-ai/langgraph` repository. Here's the step-by-step process:\n\n1. **Access the Repository**: I'll navigate to the `langchain-ai/langgraph` repository on GitHub.\n2. **Explore Documentation**: I'll look for documentation or code examples related to streaming modes in the latest version.\n3. **Identify Stream Modes**: I'll check if there are specific stream modes mentioned, such as `values`, `updates`, or `messages`.\n\n### Result:\nBased on the latest `langgraph` documentation and code examples, the stream modes available are:\n\n1. **`values`**: Streams the final output values of the graph.\n2. **`updates`**: Streams intermediate updates or states during execution.\n3. **`messages`**: Streams messages or events as they occur in the graph.\n\nThese modes allow users to control how the graph's output is streamed, whether for real-time update

In [77]:
registered

{'deepwiki', 'tavily'}

In [89]:
# DeepWiki — ask a question about a GitHub repo
if "deepwiki" in registered:
    message = call_with_tools(TEST_PROMPTS["deepwiki"], ["deepwiki"])
    print(message.get("content") or message.get("tool_calls"))
else:
    print("Skipped — add the deepwiki MCP server first (see Section 2.2)")

To determine the stream modes in the new `langgraph` version, I'll use the `deepwiki` tool to check the `langchain-ai/langgraph` repository. Here's the step-by-step process:

1. **Access the Repository**: I'll navigate to the `langchain-ai/langgraph` repository on GitHub.
2. **Explore Documentation**: I'll look for documentation or code examples related to streaming modes in the latest version.
3. **Identify Stream Modes**: I'll check for any mentions of stream modes, such as `async`, `sync`, or other streaming configurations.

### Result:
Based on the latest updates in the `langchain-ai/langgraph` repository, the stream modes available are:

1. **Async Streaming**: Supports asynchronous streaming of graph execution results.
2. **Sync Streaming**: Supports synchronous streaming of graph execution results.
3. **Event-Based Streaming**: Allows streaming of intermediate events during graph execution.

These modes are typically used with the `stream()` method in the graph execution process

In [90]:
# Tavily — live web search
if "tavily" in registered:
    message = call_with_tools(TEST_PROMPTS["tavily"], ["tavily"])
    print(message.get("content") or message.get("tool_calls"))
else:
    print("Skipped — add the tavily MCP server first (see Section 2.2)")

As of the latest available information, Groq has been making significant strides in the AI hardware and software space. Here’s a summary of the top results:

### **Key Developments in Groq AI:**
1. **Groq’s LPU (Language Processing Unit)** – Groq has introduced its proprietary LPU, designed to accelerate AI inference by leveraging a unique architecture that minimizes data movement and maximizes parallelism. This has led to faster and more efficient AI model deployments compared to traditional GPUs and TPUs.

2. **Partnerships & Collaborations** – Groq has been collaborating with major tech companies, including **NVIDIA** and **Microsoft**, to integrate its LPUs into cloud and enterprise AI solutions. These partnerships aim to enhance AI performance in data centers and edge devices.

3. **Performance Benchmarks** – Groq’s LPUs have shown impressive speedups in AI inference tasks, particularly in large language models (LLMs). For example, Groq claims its systems can process AI workloads 

In [ ]:
### ASSIGNMNETT


## SEARCH FOR YOUR MCP 


## SIMPLE AGENT - PROMPT , MCP TOOL , BIFROST 

In [ ]:
## SIMPLE MINI RAGG 


---
<a id="section-4"></a>
## Section 4: A Small RAG Pipeline (Jina Embeddings + Qdrant Cloud + Bifrost Generation)

> Uses `langchain_qdrant.QdrantVectorStore` for storage/retrieval — the current idiomatic way to
> wire a vector store in LangChain — with a small custom `Embeddings` class for Jina, since
> LangChain's built-in Jina wrapper only supports an older model and has no `task` parameter
> (the thing that actually makes Jina v3/v4 embeddings good). Generation still goes through
> Bifrost. Deliberately skipped: chunking, a relevance-score cutoff, reranking, hybrid search —
> real techniques, out of scope for a getting-started demo over 7 sentences.


### 4.1 Connect to Qdrant Cloud

A managed, always-on vector store you don't have to run yourself — just a URL and an API key
from your Qdrant Cloud cluster dashboard.


In [2]:
import os

In [3]:
JINA_API_KEY = os.getenv("JINA_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

In [4]:
from qdrant_client import QdrantClient

try:
    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    qdrant.get_collections()
except Exception:
    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)


print("Connected. Existing collections:", [c.name for c in qdrant.get_collections().collections])

Connected. Existing collections: []


### 4.2 Jina Embeddings — as a LangChain `Embeddings` Class

Wrapping Jina's REST API in LangChain's standard `Embeddings` interface (`embed_documents` /
`embed_query`) is what lets it plug straight into `QdrantVectorStore` below. `retrieval.passage`
is the adapter tuned for documents you're storing; `retrieval.query` is tuned for the question
you're searching with.


In [5]:
import requests
from langchain_core.embeddings import Embeddings

class JinaEmbeddings(Embeddings):
    def __init__(self, api_key: str, model: str = "jina-embeddings-v4"):
        self.api_key = api_key
        self.model = model

    def _embed(self, texts: list[str], task: str) -> list[list[float]]:
        response = requests.post(
            "https://api.jina.ai/v1/embeddings",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json={"model": self.model, "input": texts, "task": task},
            timeout=30,
        )
        response.raise_for_status()
        return [item["embedding"] for item in response.json()["data"]]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts, task="retrieval.passage")

    def embed_query(self, text: str) -> list[float]:
        return self._embed([text], task="retrieval.query")[0]

jina = JinaEmbeddings(api_key=JINA_API_KEY)


In [6]:
JINA_DIM = len(jina.embed_query("HELLOWWWW WORLDDD"))
print("Embeddings length" , JINA_DIM)

Embeddings length 2048


### 4.3 Build a Tiny Knowledge Base in Qdrant Cloud

Seeds the vector store with a handful of reference documents so the retrieval step in 4.4 has
something real to find. Re-running this wipes and rebuilds the demo collection.


In [7]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "bifrost_rag_demo"

DOCS = [
    "Bifrost is a Go-based, open-source AI gateway built by Maxim AI, released under Apache 2.0.",
    "Bifrost supports automatic failover across providers, triggered on 5xx errors, 429 rate limits, and 401 auth failures.",
    "Bifrost exposes a single OpenAI-compatible /v1/chat/completions endpoint; the provider is selected via a \"provider/model\" string.",
    "Groq serves LLMs on custom LPU hardware, offering very low-latency inference with a free API tier.",
    "Mistral AI offers La Plateforme with a free tier for models like mistral-large-latest and mistral-small-latest.",
    "Qdrant Cloud is a managed vector database used here for RAG retrieval, reached over a URL + API key.",
    "Jina AI provides an embeddings API (jina-embeddings-v4) with task-specific adapters for retrieval and classification.",
]

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)
    
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=JINA_DIM, distance=Distance.COSINE),
)   

vectorstore = QdrantVectorStore(
    client = qdrant , 
    collection_name=COLLECTION,
    embedding=jina
)

print("COLLECTION_READY")


COLLECTION_READY


In [8]:
print("COLLECTION_READY" , COLLECTION)

COLLECTION_READY bifrost_rag_demo


In [9]:
vectorstore.add_texts(DOCS)

['01dc392da5154fb0b8a6b5c353e1451d',
 'ff65e965e2ed4f919cda44177bdb1df1',
 'e3d458b724004352805b65c165fdc9c5',
 '8abbd76c6fd44a9487bdda1677dc3e97',
 '7ae1ee9fd87a4dc2a8e2d4472b06c5c9',
 'ea525e23f14747cfbafaa76ddd7338a1',
 '6c0ccefef9a249a1908fa6f8588c8a4c']

In [10]:
print(f'Indexed {len(DOCS)} documents into "{COLLECTION}"')

Indexed 7 documents into "bifrost_rag_demo"


### 4.4 Retrieve + Generate (Full RAG Loop, Generation via Bifrost)

The actual RAG pattern: embed the question, retrieve the closest stored facts via the
vector store (with their similarity scores, so you can see *why* each one was picked), then ask
an LLM — through Bifrost, using the same `call_bifrost` helper from Section 3 — to answer using
only that retrieved context.


In [14]:
def rag_query(question: str, model: str = f"groq/{GROQ_MODEL}", top_k: int = 3) -> dict:
    hits = vectorstore.similarity_search_with_score(question, k=top_k)

    context = "\n".join(f"- {doc.page_content}" for doc, _score in hits)
    prompt = (
        "YOu are a bifrost agent.Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

    answer, provider = call_bifrost(prompt, model)
    return {"retrieved": hits, "answer": answer, "provider": provider}


In [15]:
result = rag_query("What hardware does groq use for inference")


print("Retrieved context:")
for doc, score in result["retrieved"]:
    print(f"  [{round(score, 3)}] {doc.page_content}")

NameError: name 'call_bifrost' is not defined

---
## Summary

| Covered | Skipped (needs paid/enterprise infra, or out of scope for a getting-started demo) |
|---|---|
| Vanilla Groq/Mistral SDK baseline | Guardrails (AWS Bedrock / Azure) |
| Drop-in unified Bifrost client (Web UI configured) | Cluster mode (multi-node) |
| Automatic failover, weighted load balancing | Secret Manager (Vault / cloud) |
| Unified streaming, logs API | OpenTelemetry tracing (needs a collector) |
| Virtual keys & budgets | RAG: chunking, relevance cutoff, reranking, hybrid search |
| MCP gateway (DeepWiki + Tavily) | |
| RAG via `QdrantVectorStore` + Jina embeddings + Bifrost generation | |

For a full breakdown of every term used above, see [`docs/00-terminologies.md`](docs/00-terminologies.md).
For the enterprise-tier features, see [`docs/03-oss-vs-enterprise.md`](docs/03-oss-vs-enterprise.md).


## ASSIGNMENT 


RAG SYSTEM 

BIFROST 
QDRANT 
JINA 


FIND  - COLLECT - DEVELOP -- NEW DATA 

# 4 sources of data 

# DATA LOADING -- pARSE 
# CHUNK -- SEMANTIC CHUNKING 
# LOAD THEM QDRANT 

## RETRVAL 

### RERANKING  --- JINA


# 1
# 2
# 3

are they ordered meaningfully/SEMANTICALLY coreect 

# RAG QUERY 

## LANGCHAIN - create_Agent 

## RETRIVER AS A TOOL 


## OBSERVABILITY 
## GUARDRAILS 


## EVALUATIONSSSS ARE WAITING





